# LBM D2Q9 — GPU build, tests and benchmarks on Google Colab

Runs the CUDA versions on a Colab GPU (Tesla T4, `sm_75`): build in float and double, run all
tests, check the kernels with `compute-sanitizer`, profile them with Nsight Compute, measure the
GPU memory bandwidth and benchmark the solvers. The CSV files produced in `results/` are then
downloaded and committed to the repository.

**Before running:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

GPU clocks on a shared cloud machine vary between sessions (about 5% observed), so versions
that are compared with each other are always measured in the same session.

In [ ]:
# What to run in this session. Results already committed are not redone by default.
RUN_V2_BENCH = False      # step 3: v2 benchmarks
RUN_V3_BENCH = False      # step 4: bandwidth, Nsight Compute, v3 benchmarks
RUN_CYLINDER = True       # step 5: cylinder flow with v3, Strouhal number

## 1. Environment (GPU, driver, CUDA toolkit)

In [ ]:
!nvidia-smi
!nvcc --version
!cmake --version | head -1
!nvidia-smi --query-gpu=name,driver_version,ecc.mode.current,clocks.max.sm,clocks.max.memory --format=csv

## 2. Get the code

In [ ]:
REPO = "https://github.com/H-HABCHANE/Simulation-de-fluide-par-la-m-thode-de-Boltzmann-sur-r-seau-LBM-du-CPU-au-multi-GPU.git"
!rm -rf lbm && git clone --quiet {REPO} lbm
%cd lbm
!git log --oneline -1

## 3. Build (float and double) and run all tests

In [ ]:
for prec in ["OFF", "ON"]:
    d = "build-double" if prec == "ON" else "build-float"
    !cmake -S . -B {d} -DLBM_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLBM_REAL_DOUBLE={prec} > /dev/null
    !cmake --build {d} -j 4 2>&1 | grep -Ei "warning|error" || true
    !ctest --test-dir {d} --output-on-failure

## 4. `compute-sanitizer` on every CUDA version

Each tool must report `ERROR SUMMARY: 0 errors`: `memcheck` (out-of-bounds and misaligned
accesses), `initcheck` (reads of uninitialised device memory), `racecheck` (shared-memory
races), `synccheck` (invalid synchronisation).

In [ ]:
for test in ["test_v2_vs_v0", "test_v3_vs_v0", "test_cylinder_v3"]:
    for tool in ["memcheck", "initcheck", "racecheck", "synccheck"]:
        print(f"===== {test}: {tool} =====")
        !compute-sanitizer --tool {tool} --error-exitcode 1 build-float/tests/{test} 2>&1 | tail -2

## 5. GPU memory bandwidth and MLUPS ceiling

In [ ]:
if RUN_V3_BENCH:
    !mkdir -p results
    !build-float/gpu_bandwidth results/gpu_bandwidth_Tesla-T4.csv
    # 320 GB/s: Tesla T4 datasheet value.
    !python3 bench/ceiling.py results/gpu_bandwidth_Tesla-T4.csv --gpu --peak-gbs 320

## 6. Step 3 benchmarks of v2 (already committed; `RUN_V2_BENCH` to redo them)


In [ ]:
if RUN_V2_BENCH:
    SIZES = "128x128 512x512 1024x1024 2048x2048 4096x4096"
    !python3 bench/run_bench.py --exe build-float/src/v2_cuda_naive/lbm_v2 --sizes 2048x2048 --blocks 32x8 16x16 64x4 128x2 32x16 --updates 5e8 --label v2_blocks --csv results/v2_blocks_Tesla-T4.csv
    !python3 bench/run_bench.py --exe build-float/src/v2_cuda_naive/lbm_v2 --sizes {SIZES} --updates 5e8
    !python3 bench/run_bench.py --exe build-double/src/v2_cuda_naive/lbm_v2 --sizes 1024x1024 2048x2048 --updates 2e8 --label v2_double --csv results/v2_double_Tesla-T4.csv

## 7. Step 4 — Nsight Compute: real memory traffic of v2 and of each v3 variant

Bytes per cell update measured in DRAM (ideal: 72 in float) and sectors per request
(4 = fully coalesced warp access). If the GPU counters are blocked on this runtime
(`ERR_NVGPUCTRPERM`), this cell reports it and the analysis falls back on the effective
bandwidth (MLUPS x 72 bytes).

In [ ]:
if RUN_V3_BENCH:
    !rm -f results/ncu_Tesla-T4.csv
    !python3 bench/ncu_profile.py --exe build-float/src/v2_cuda_naive/lbm_v2 --label v2 --csv results/ncu_Tesla-T4.csv
    for v in ["soa_push", "soa_pull", "soa_pull_opt"]:
        !python3 bench/ncu_profile.py --exe build-float/src/v3_cuda_opt/lbm_v3 --variant {v} --label v3_{v} --csv results/ncu_Tesla-T4.csv
    !cat results/ncu_Tesla-T4.csv

## 8. Step 4 — benchmarks, all in this session

1. v2 and the three v3 variants on every grid size (optimisation steps, same session);
2. block size sweep of the final variant;
3. double precision;
4. FMA contraction allowed (`LBM_CUDA_FMAD=ON`), to measure its effect only.

In [ ]:
if RUN_V3_BENCH:
    SIZES = "128x128 512x512 1024x1024 2048x2048 4096x4096"
    STEPS_CSV = "results/v3_steps_Tesla-T4.csv"
    !rm -f {STEPS_CSV}
    !python3 bench/run_bench.py --exe build-float/src/v2_cuda_naive/lbm_v2 --sizes {SIZES} --updates 5e8 --csv {STEPS_CSV}
    !python3 bench/run_bench.py --exe build-float/src/v3_cuda_opt/lbm_v3 --sizes {SIZES} --variants soa_push soa_pull soa_pull_opt --updates 5e8 --csv {STEPS_CSV}

In [ ]:
if RUN_V3_BENCH:
    !python3 bench/run_bench.py --exe build-float/src/v3_cuda_opt/lbm_v3 --sizes 2048x2048 --blocks 32x8 64x4 128x2 32x16 16x16 256x1 --updates 5e8 --label v3_blocks --csv results/v3_blocks_Tesla-T4.csv
    !python3 bench/run_bench.py --exe build-double/src/v2_cuda_naive/lbm_v2 --sizes 2048x2048 --updates 2e8 --label v2_double --csv results/v3_double_Tesla-T4.csv
    !python3 bench/run_bench.py --exe build-double/src/v3_cuda_opt/lbm_v3 --sizes 2048x2048 --updates 2e8 --label v3_double --csv results/v3_double_Tesla-T4.csv

In [ ]:
if RUN_V3_BENCH:
    !cmake -S . -B build-fmad -DLBM_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLBM_CUDA_FMAD=ON > /dev/null && cmake --build build-fmad -j 4 --target lbm_v3 > /dev/null
    !python3 bench/run_bench.py --exe build-float/src/v3_cuda_opt/lbm_v3 --sizes 2048x2048 4096x4096 --updates 5e8 --label v3 --csv results/v3_fmad_Tesla-T4.csv
    !python3 bench/run_bench.py --exe build-fmad/src/v3_cuda_opt/lbm_v3 --sizes 2048x2048 4096x4096 --updates 5e8 --label v3_fmad --csv results/v3_fmad_Tesla-T4.csv

## Step 5 — flow past a cylinder (Re = 100) with v3

1000 x 320 grid, D = 40, inlet velocity 0.05 (Ma = 0.087), tau = 0.56, 160 000 steps (same run as v1). The
Strouhal number is computed from the velocity probe 3 D behind the cylinder and appended to
`results/strouhal.csv`; it must agree with the CPU run (v1) of the same case.

In [ ]:
if RUN_CYLINDER:
    !build-float/src/v3_cuda_opt/lbm_v3 --case cylinder --nx 1000 --ny 320 --diameter 40 --re 100 --umax 0.05 --steps 160000 --probe-every 50 --out output/cylinder_v3
    !python3 scripts/strouhal.py output/cylinder_v3 --from 0.55 --label v3_Tesla-T4

## Download the results (to be copied into `results/` and committed)


In [ ]:
!ls -l results/
!cd results && zip -q ../results_colab.zip *.csv
from google.colab import files
files.download("results_colab.zip")